# Unified Multi-Honeypot Data Merging

---

## Dataset Overview
* **Sources:** Cowrie SSH, Tanner HTTP, Dionaea FTP, and T-Pot FTP logs.
* **Protocols:** SSH, HTTP, and FTP.
* **Goal:** Merge 4 different honeypot datasets into 1 single standardized dataset.

---

## Processing Steps
1. **Schema Setup:** Defined a single 24-column structure for all datasets.
2. **Normalization:** Converted and mapped columns from each honeypot to match the standard structure.
3. **Data Merging:** Combined all 4 datasets into one unified dataset.
4. **Deduplication:** Removed exact duplicate rows to keep the data clean.

---

## Final Output
* **File:** `unified_security_events_clean_v1.csv` – A single, clean dataset containing all merged honeypot logs.

In [ ]:
import pandas as pd

# Load datasets
ftp1 = pd.read_csv("ftp_clean_final.csv")
ftp2 = pd.read_csv("dionaea_ftp_clean.csv")

print("FTP Dataset 1")
print("Shape:", ftp1.shape)
print()

print("Dionaea FTP Dataset")
print("Shape:", ftp2.shape)

FTP Dataset 1
Shape: (76, 36)

Dionaea FTP Dataset
Shape: (1344, 13)


In [ ]:
# Compare columns

cols1 = set(ftp1.columns)
cols2 = set(ftp2.columns)

print("Columns in ftp_clean_final.csv:")
print(sorted(cols1))
print()

print("Columns in dionaea_ftp_clean.csv:")
print(sorted(cols2))
print()

print("Common columns:")
print(sorted(cols1 & cols2))
print()

print("Only in ftp_clean_final.csv:")
print(sorted(cols1 - cols2))
print()

print("Only in dionaea_ftp_clean.csv:")
print(sorted(cols2 - cols1))

Columns in ftp_clean_final.csv:
['@timestamp', 'command_count', 'dest_ip', 'geoip.asn', 'geoip.city_name', 'geoip.country_code2', 'geoip.region_code', 'geoip_ext.city_name', 'geoip_ext.continent_code', 'geoip_ext.country_code2', 'geoip_ext.country_code3', 'geoip_ext.country_name', 'geoip_ext.dma_code', 'geoip_ext.ip', 'geoip_ext.latitude', 'geoip_ext.location.lat', 'geoip_ext.location.lon', 'geoip_ext.longitude', 'geoip_ext.postal_code', 'geoip_ext.region_code', 'geoip_ext.region_name', 'geoip_ext.timezone', 'has_AUTH', 'has_PASS', 'has_USER', 'has_password', 'has_username', 'headers.content_length', 'host', 'ip_rep', 'is_anonymous', 'src_ip', 'src_port', 't-pot_hostname', 't-pot_ip_ext', 't-pot_ip_int']

Columns in dionaea_ftp_clean.csv:
['connection', 'connection_protocol', 'connection_timestamp', 'date', 'hour', 'local_port', 'login', 'login_password', 'login_username', 'remote_host', 'remote_port', 'timestamp', 'weekday']

Common columns:
[]

Only in ftp_clean_final.csv:
['@timesta

In [ ]:
# Data types comparison

profile1 = pd.DataFrame({
    "dtype": ftp1.dtypes.astype(str),
    "non_null": ftp1.notna().sum(),
    "null": ftp1.isna().sum(),
    "unique": ftp1.nunique(dropna=True)
})

profile2 = pd.DataFrame({
    "dtype": ftp2.dtypes.astype(str),
    "non_null": ftp2.notna().sum(),
    "null": ftp2.isna().sum(),
    "unique": ftp2.nunique(dropna=True)
})

print("=== FTP 1 Profile ===")
display(profile1)

print("=== Dionaea FTP Profile ===")
display(profile2)

=== FTP 1 Profile ===


,dtype,non_null,null,unique
t-pot_hostname,object,76,0,4
t-pot_ip_int,object,76,0,4
t-pot_ip_ext,object,76,0,10
src_port,int64,76,0,75
@timestamp,object,76,0,75
host,object,76,0,2
dest_ip,object,76,0,4
src_ip,object,76,0,53
geoip_ext.timezone,object,76,0,4
geoip_ext.latitude,float64,76,0,4


=== Dionaea FTP Profile ===


,dtype,non_null,null,unique
connection,int64,1344,0,186
connection_protocol,object,1344,0,1
connection_timestamp,float64,1344,0,187
local_port,int64,1344,0,1
remote_host,object,1344,0,104
remote_port,int64,1344,0,184
timestamp,object,1344,0,187
date,object,1344,0,26
login,float64,1335,9,1294
login_username,object,1335,9,34


In [ ]:
# ==========================================
# FTP FIELD VALIDATION
# ==========================================

import pandas as pd

ftp1 = pd.read_csv("ftp_clean_final.csv")
ftp2 = pd.read_csv("dionaea_ftp_clean.csv")

print("========== FTP DATASET 1 ==========")
print(ftp1[
    [
        "src_ip",
        "src_port",
        "dest_ip",
        "@timestamp",
        "command_count",
        "has_USER",
        "has_PASS",
        "has_AUTH",
        "has_username",
        "has_password",
        "is_anonymous"
    ]
].head(10).to_string())

print("\n========== DIONAEA FTP ==========")
print(ftp2[
    [
        "connection",
        "connection_protocol",
        "connection_timestamp",
        "local_port",
        "remote_host",
        "remote_port",
        "timestamp",
        "login",
        "login_username",
        "login_password"
    ]
].head(10).to_string())

========== FTP DATASET 1 ==========
           src_ip  src_port     dest_ip                @timestamp  command_count  has_USER  has_PASS  has_AUTH  has_username  has_password  is_anonymous
0   198.235.24.69     51168  172.21.0.2  2025-06-11T15:33:07.648Z              1         0         0         1             0             0             0
1   106.75.50.248     37444  172.20.0.2  2025-06-11T15:27:12.437Z              1         0         0         1             0             0             0
2   106.75.50.248     36978  172.20.0.2  2025-06-11T15:27:11.970Z              1         1         0         0             0             0             0
3      20.38.33.1     53820  172.20.0.2  2025-06-11T15:10:03.296Z              2         0         0         1             0             0             0
4   20.65.193.113     38820  172.21.0.2  2025-06-11T15:00:35.116Z              2         0         0         1             0             0             0
5   20.65.193.113     60182  172.19.0.2  2025-

In [ ]:
# ==========================================
# VALUE / RELATIONSHIP CHECK
# ==========================================

print("========== FTP 1 ==========")

print("\nProtocol / Ports:")
print("src_port unique:", ftp1["src_port"].nunique())
print("dest_ip unique:", ftp1["dest_ip"].nunique())

print("\nAuthentication indicators:")
print(ftp1[
    ["has_USER", "has_PASS", "has_AUTH",
     "has_username", "has_password", "is_anonymous"]
].sum())

print("\nCommand count:")
print(ftp1["command_count"].describe())


print("\n========== DIONAEA ==========")

print("\nConnection:")
print("Unique connections:", ftp2["connection"].nunique())
print("Rows:", len(ftp2))

print("\nProtocol:")
print(ftp2["connection_protocol"].value_counts(dropna=False))

print("\nPorts:")
print("Local ports:")
print(ftp2["local_port"].value_counts(dropna=False))

print("\nRemote ports:")
print(ftp2["remote_port"].describe())

print("\nLogin:")
print("login non-null:", ftp2["login"].notna().sum())
print("username non-null:", ftp2["login_username"].notna().sum())
print("password non-null:", ftp2["login_password"].notna().sum())

print("\nUnique usernames:",
      ftp2["login_username"].nunique(dropna=True))

print("Unique passwords:",
      ftp2["login_password"].nunique(dropna=True))

========== FTP 1 ==========

Protocol / Ports:
src_port unique: 75
dest_ip unique: 4

Authentication indicators:
has_USER        11
has_PASS         7
has_AUTH        29
has_username     4
has_password     4
is_anonymous     4
dtype: int64

Command count:
count    76.000000
mean      1.197368
std       1.366453
min       0.000000
25%       0.000000
50%       1.000000
75%       2.000000
max       7.000000
Name: command_count, dtype: float64

========== DIONAEA ==========

Connection:
Unique connections: 186
Rows: 1344

Protocol:
connection_protocol
ftpd    1344
Name: count, dtype: int64

Ports:
Local ports:
local_port
21    1344
Name: count, dtype: int64

Remote ports:
count     1344.000000
mean     46642.666667
std      14869.582561
min        221.000000
25%      40656.000000
50%      50252.000000
75%      58064.000000
max      65520.000000
Name: remote_port, dtype: float64

Login:
login non-null: 1335
username non-null: 1335
password non-null: 1315

Unique usernames: 34
Unique passwor

In [ ]:
print(
    ftp2.groupby("connection").agg(
        rows=("connection", "size"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        unique_users=("login_username", "nunique"),
        unique_passwords=("login_password", "nunique"),
        unique_remote_hosts=("remote_host", "nunique"),
        unique_remote_ports=("remote_port", "nunique")
    ).head(20)
)

            rows                first_timestamp  \
connection                                        
3              1  2020-02-02 16:34:36.961342812   
5              2  2020-02-10 03:39:50.386400700   
122            2  2020-02-14 04:21:26.532921314   
133            3  2020-02-19 03:59:04.878040075   
233            3  2020-02-21 04:36:01.848979473   
262            2  2020-02-10 04:44:44.541862488   
294            1  2020-02-08 04:46:19.025120258   
325            3  2020-02-20 04:48:53.368005514   
334            4  2020-01-23 04:54:58.202168465   
400            3  2020-02-09 17:27:58.860017538   
547            6  2020-02-20 05:28:45.279347420   
556            4  2020-02-14 05:43:28.101692438   
576            6  2020-02-13 06:01:56.201093674   
625            4  2020-02-14 05:57:56.394268036   
626            5  2020-02-14 05:57:57.927052259   
627            4  2020-02-14 05:57:59.456721306   
639            3  2020-02-15 05:48:27.558816671   
647            5  2020-01-26 17

In [ ]:
import pandas as pd
import numpy as np

ftp1 = pd.read_csv("ftp_clean_final.csv")
dionaea = pd.read_csv("dionaea_ftp_clean.csv")

print("FTP 1:", ftp1.shape)
print("Dionaea:", dionaea.shape)

FTP 1: (76, 36)
Dionaea: (1344, 13)


In [ ]:
UNIFIED_FTP_COLUMNS = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "command_count",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "source_dataset"
]

In [ ]:
ftp1_norm = pd.DataFrame(index=ftp1.index)

ftp1_norm["timestamp"] = pd.to_datetime(
    ftp1["@timestamp"],
    utc=True,
    errors="coerce"
)

ftp1_norm["honeypot"] = "t-pot-ftp"
ftp1_norm["protocol"] = "FTP"

ftp1_norm["source_ip"] = ftp1["src_ip"]
ftp1_norm["source_port"] = ftp1["src_port"]

ftp1_norm["destination_ip"] = ftp1["dest_ip"]

# Destination port is not explicitly available
ftp1_norm["destination_port"] = np.nan

# No explicit session ID
ftp1_norm["session_id"] = np.nan

ftp1_norm["command_count"] = ftp1["command_count"]

# These are indicators, NOT actual credential values
ftp1_norm["username"] = np.nan
ftp1_norm["password"] = np.nan

ftp1_norm["user_attempt"] = ftp1["has_USER"]
ftp1_norm["pass_attempt"] = ftp1["has_PASS"]
ftp1_norm["auth_attempt"] = ftp1["has_AUTH"]

ftp1_norm["anonymous"] = ftp1["is_anonymous"]

ftp1_norm["country"] = ftp1["geoip_ext.country_name"]
ftp1_norm["country_code"] = ftp1["geoip_ext.country_code2"]
ftp1_norm["asn"] = ftp1["geoip.asn"]

ftp1_norm["ip_reputation"] = ftp1["ip_rep"]

ftp1_norm["source_dataset"] = "ftp_clean_final"

In [ ]:
dionaea_norm = pd.DataFrame(index=dionaea.index)

dionaea_norm["timestamp"] = pd.to_datetime(
    dionaea["timestamp"],
    utc=True,
    errors="coerce"
)

dionaea_norm["honeypot"] = "dionaea"
dionaea_norm["protocol"] = "FTP"

dionaea_norm["source_ip"] = dionaea["remote_host"]
dionaea_norm["source_port"] = dionaea["remote_port"]

# Dionaea does not provide destination IP
dionaea_norm["destination_ip"] = np.nan

dionaea_norm["destination_port"] = dionaea["local_port"]

dionaea_norm["session_id"] = dionaea["connection"]

# No command count in Dionaea dataset
dionaea_norm["command_count"] = np.nan

dionaea_norm["username"] = dionaea["login_username"]
dionaea_norm["password"] = dionaea["login_password"]

# These command-level indicators are not present
dionaea_norm["user_attempt"] = np.nan
dionaea_norm["pass_attempt"] = np.nan
dionaea_norm["auth_attempt"] = np.nan

# Do not infer anonymous behavior without evidence
dionaea_norm["anonymous"] = np.nan

# No GeoIP enrichment in this dataset
dionaea_norm["country"] = np.nan
dionaea_norm["country_code"] = np.nan
dionaea_norm["asn"] = np.nan
dionaea_norm["ip_reputation"] = np.nan

dionaea_norm["source_dataset"] = "dionaea_ftp_clean"

In [ ]:
print("FTP 1 normalized:")
display(ftp1_norm.head())

print("\nDionaea normalized:")
display(dionaea_norm.head())

print("\nColumns identical:",
      list(ftp1_norm.columns) == list(dionaea_norm.columns))

FTP 1 normalized:


,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,command_count,username,password,user_attempt,pass_attempt,auth_attempt,anonymous,country,country_code,asn,ip_reputation,source_dataset
0,2025-06-11 15:33:07.648000+00:00,t-pot-ftp,FTP,198.235.24.69,51168,172.21.0.2,NaN,NaN,1,NaN,NaN,0,0,1,0,South Africa,ZA,396982,NaN,ftp_clean_final
1,2025-06-11 15:27:12.437000+00:00,t-pot-ftp,FTP,106.75.50.248,37444,172.20.0.2,NaN,NaN,1,NaN,NaN,0,0,1,0,United States,US,4808,NaN,ftp_clean_final
2,2025-06-11 15:27:11.970000+00:00,t-pot-ftp,FTP,106.75.50.248,36978,172.20.0.2,NaN,NaN,1,NaN,NaN,1,0,0,0,United States,US,4808,NaN,ftp_clean_final
3,2025-06-11 15:10:03.296000+00:00,t-pot-ftp,FTP,20.38.33.1,53820,172.20.0.2,NaN,NaN,2,NaN,NaN,0,0,1,0,United States,US,8075,NaN,ftp_clean_final
4,2025-06-11 15:00:35.116000+00:00,t-pot-ftp,FTP,20.65.193.113,38820,172.21.0.2,NaN,NaN,2,NaN,NaN,0,0,1,0,South Africa,ZA,8075,known attacker,ftp_clean_final



Dionaea normalized:


,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,command_count,username,password,user_attempt,pass_attempt,auth_attempt,anonymous,country,country_code,asn,ip_reputation,source_dataset
0,2020-02-15 05:48:27.558816671+00:00,dionaea,FTP,192.241.231.243,56190,NaN,21,639,NaN,sa,dddddddd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
1,2020-02-15 05:48:27.558816671+00:00,dionaea,FTP,192.241.231.243,56190,NaN,21,639,NaN,sa,1qazxsw2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
2,2020-02-15 05:48:27.558816671+00:00,dionaea,FTP,192.241.231.243,56190,NaN,21,639,NaN,sa,0123456789,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
3,2020-02-15 16:36:56.719931841+00:00,dionaea,FTP,104.206.128.30,63509,NaN,21,20231,NaN,sa,feifei521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
4,2020-02-15 16:36:56.719931841+00:00,dionaea,FTP,104.206.128.30,63509,NaN,21,20231,NaN,sa,19890125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean



Columns identical: True


In [ ]:
print("FTP 1 normalized shape:", ftp1_norm.shape)
print("Dionaea normalized shape:", dionaea_norm.shape)

FTP 1 normalized shape: (76, 20)
Dionaea normalized shape: (1344, 20)


In [ ]:
%whos DataFrame

Variable       Type         Data/Info
-------------------------------------
dionaea        DataFrame          connection connecti<...>n[1344 rows x 13 columns]
dionaea_norm   DataFrame                             <...>n[1344 rows x 20 columns]
ftp1           DataFrame             t-pot_hostname t<...>n\n[76 rows x 36 columns]
ftp1_norm      DataFrame                             <...>n\n[76 rows x 20 columns]
ftp2           DataFrame          connection connecti<...>n[1344 rows x 13 columns]
profile1       DataFrame                             <...>4        76     0       2
profile2       DataFrame                            d<...>t      1344     0       7


In [ ]:
ftp_unified = pd.concat(
    [ftp1_norm, dionaea_norm],
    ignore_index=True
)

print("FTP Unified Shape:", ftp_unified.shape)
print("\nColumns:")
print(ftp_unified.columns.tolist())

print("\nHoneypot Distribution:")
print(ftp_unified["honeypot"].value_counts())

print("\nSource Dataset Distribution:")
print(ftp_unified["source_dataset"].value_counts())

FTP Unified Shape: (1420, 20)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'command_count', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'country', 'country_code', 'asn', 'ip_reputation', 'source_dataset']

Honeypot Distribution:
honeypot
dionaea      1344
t-pot-ftp      76
Name: count, dtype: int64

Source Dataset Distribution:
source_dataset
dionaea_ftp_clean    1344
ftp_clean_final        76
Name: count, dtype: int64


In [ ]:
print("Shape:", ftp_unified.shape)

print("\nMissing Values:")
print(ftp_unified.isnull().sum())

print("\nData Types:")
print(ftp_unified.dtypes)

print("\nProtocols:")
print(ftp_unified["protocol"].value_counts())

print("\nHoneypots:")
print(ftp_unified["honeypot"].value_counts())

print("\nDuplicates:", ftp_unified.duplicated().sum())

Shape: (1420, 20)

Missing Values:
timestamp              0
honeypot               0
protocol               0
source_ip              0
source_port            0
destination_ip      1344
destination_port      76
session_id            76
command_count       1344
username              85
password             105
user_attempt        1344
pass_attempt        1344
auth_attempt        1344
anonymous           1344
country             1344
country_code        1344
asn                 1344
ip_reputation       1382
source_dataset         0
dtype: int64

Data Types:
timestamp           datetime64[ns, UTC]
honeypot                         object
protocol                         object
source_ip                        object
source_port                       int64
destination_ip                   object
destination_port                float64
session_id                      float64
command_count                   float64
username                         object
password                         object

In [ ]:
# Check exact duplicate rows

duplicates = ftp_unified[
    ftp_unified.duplicated(keep=False)
].sort_values(
    by=["timestamp", "source_dataset"]
)

print("Number of duplicate rows:", len(duplicates))

duplicates

Number of duplicate rows: 6


,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,command_count,username,password,user_attempt,pass_attempt,auth_attempt,anonymous,country,country_code,asn,ip_reputation,source_dataset
734,2020-01-26 08:18:57.813625336+00:00,dionaea,FTP,80.82.70.118,60000,NaN,21.0,14391.0,NaN,sa,19840517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
736,2020-01-26 08:18:57.813625336+00:00,dionaea,FTP,80.82.70.118,60000,NaN,21.0,14391.0,NaN,sa,19840517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
120,2020-02-14 09:06:40.307245255+00:00,dionaea,FTP,71.6.232.4,33694,NaN,21.0,1763.0,NaN,sa,19881020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
121,2020-02-14 09:06:40.307245255+00:00,dionaea,FTP,71.6.232.4,33694,NaN,21.0,1763.0,NaN,sa,19881020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
45,2025-06-10 01:41:32.388000+00:00,t-pot-ftp,FTP,205.210.31.12,62076,172.19.0.2,NaN,NaN,2.0,NaN,NaN,0.0,0.0,1.0,0.0,Spain,ES,396982.0,NaN,ftp_clean_final
46,2025-06-10 01:41:32.388000+00:00,t-pot-ftp,FTP,205.210.31.12,62076,172.19.0.2,NaN,NaN,2.0,NaN,NaN,0.0,0.0,1.0,0.0,Spain,ES,396982.0,NaN,ftp_clean_final


In [ ]:
# Count duplicated records by source dataset

print(
    duplicates["source_dataset"]
    .value_counts()
)

source_dataset
dionaea_ftp_clean    4
ftp_clean_final      2
Name: count, dtype: int64


In [ ]:
# Check duplicates based on important event identity fields

dup_check = ftp_unified[
    ftp_unified.duplicated(
        subset=[
            "timestamp",
            "honeypot",
            "source_ip",
            "source_port",
            "destination_ip",
            "destination_port",
            "session_id",
            "username",
            "password"
        ],
        keep=False
    )
].sort_values("timestamp")

dup_check

,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,command_count,username,password,user_attempt,pass_attempt,auth_attempt,anonymous,country,country_code,asn,ip_reputation,source_dataset
734,2020-01-26 08:18:57.813625336+00:00,dionaea,FTP,80.82.70.118,60000,NaN,21.0,14391.0,NaN,sa,19840517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
736,2020-01-26 08:18:57.813625336+00:00,dionaea,FTP,80.82.70.118,60000,NaN,21.0,14391.0,NaN,sa,19840517,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
120,2020-02-14 09:06:40.307245255+00:00,dionaea,FTP,71.6.232.4,33694,NaN,21.0,1763.0,NaN,sa,19881020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
121,2020-02-14 09:06:40.307245255+00:00,dionaea,FTP,71.6.232.4,33694,NaN,21.0,1763.0,NaN,sa,19881020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dionaea_ftp_clean
45,2025-06-10 01:41:32.388000+00:00,t-pot-ftp,FTP,205.210.31.12,62076,172.19.0.2,NaN,NaN,2.0,NaN,NaN,0.0,0.0,1.0,0.0,Spain,ES,396982.0,NaN,ftp_clean_final
46,2025-06-10 01:41:32.388000+00:00,t-pot-ftp,FTP,205.210.31.12,62076,172.19.0.2,NaN,NaN,2.0,NaN,NaN,0.0,0.0,1.0,0.0,Spain,ES,396982.0,NaN,ftp_clean_final


In [ ]:
# Remove exact duplicate rows
ftp_unified = ftp_unified.drop_duplicates().reset_index(drop=True)

print("Shape after removing exact duplicates:", ftp_unified.shape)
print("Remaining duplicate rows:", ftp_unified.duplicated().sum())

Shape after removing exact duplicates: (1417, 20)
Remaining duplicate rows: 0


In [ ]:
print("=== FTP Unified Final Validation ===")

print("\nShape:")
print(ftp_unified.shape)

print("\nDuplicates:")
print(ftp_unified.duplicated().sum())

print("\nHoneypot Distribution:")
print(ftp_unified["honeypot"].value_counts())

print("\nProtocol Distribution:")
print(ftp_unified["protocol"].value_counts())

print("\nSource Dataset:")
print(ftp_unified["source_dataset"].value_counts())

print("\nMissing Values:")
print(ftp_unified.isnull().sum())

=== FTP Unified Final Validation ===

Shape:
(1417, 20)

Duplicates:
0

Honeypot Distribution:
honeypot
dionaea      1342
t-pot-ftp      75
Name: count, dtype: int64

Protocol Distribution:
protocol
FTP    1417
Name: count, dtype: int64

Source Dataset:
source_dataset
dionaea_ftp_clean    1342
ftp_clean_final        75
Name: count, dtype: int64

Missing Values:
timestamp              0
honeypot               0
protocol               0
source_ip              0
source_port            0
destination_ip      1342
destination_port      75
session_id            75
command_count       1342
username              84
password             104
user_attempt        1342
pass_attempt        1342
auth_attempt        1342
anonymous           1342
country             1342
country_code        1342
asn                 1342
ip_reputation       1379
source_dataset         0
dtype: int64


In [ ]:
import pandas as pd

cowrie = pd.read_csv("cowrie_ssh_preprocessed.csv")
tanner = pd.read_csv("tanner_http_labeled.csv")

print("COWRIE SSH ")
print("Shape:", cowrie.shape)
print("\nColumns:")
print(cowrie.columns.tolist())
print("\nData Types:")
print(cowrie.dtypes)
print("\nMissing Values:")
print(cowrie.isna().sum())
print("\nDuplicates:", cowrie.duplicated().sum())


print("\n\n TANNER HTTP")
print("Shape:", tanner.shape)
print("\nColumns:")
print(tanner.columns.tolist())
print("\nData Types:")
print(tanner.dtypes)
print("\nMissing Values:")
print(tanner.isna().sum())
print("\nDuplicates:", tanner.duplicated().sum())

/tmp/ipykernel_1475/2214552424.py:3: DtypeWarning: Columns (68,70,72) have mixed types. Specify dtype option on import or set low_memory=False.
  cowrie = pd.read_csv("cowrie_ssh_preprocessed.csv")


COWRIE SSH 
Shape: (33454, 73)

Columns:
['t-pot_hostname', 't-pot_ip_ext', 't-pot_ip_int', 'timestamp', 'message', 'eventid', 'sensor', '@timestamp', 'host', 'duration', 'src_ip', 'session', 'geoip_ext.timezone', 'geoip_ext.latitude', 'geoip_ext.postal_code', 'geoip_ext.country_code2', 'geoip_ext.country_name', 'geoip_ext.country_code3', 'geoip_ext.region_code', 'geoip_ext.as_org', 'geoip_ext.continent_code', 'geoip_ext.region_name', 'geoip_ext.city_name', 'geoip_ext.longitude', 'geoip_ext.location.lat', 'geoip_ext.location.lon', 'geoip_ext.ip', 'geoip_ext.asn', 'headers.content_length', 'geoip.timezone', 'geoip.latitude', 'geoip.postal_code', 'geoip.country_code2', 'geoip.country_name', 'geoip.country_code3', 'geoip.region_code', 'geoip.as_org', 'geoip.continent_code', 'geoip.region_name', 'geoip.city_name', 'geoip.longitude', 'geoip.location.lat', 'geoip.location.lon', 'geoip.ip', 'geoip.asn', 'dest_port', 'dest_ip', 'ip_rep', 'src_port', 'protocol', 'geoip.dma_code', 'password', 'u

In [ ]:
# ==========================================
# COWRIE SSH NORMALIZATION - FIXED
# ==========================================

import pandas as pd
import numpy as np

cowrie_norm = pd.DataFrame()

# ------------------------------------------
# Core fields
# ------------------------------------------

cowrie_norm["timestamp"] = pd.to_datetime(
    cowrie["timestamp"],
    format="mixed",
    utc=True
)

cowrie_norm["honeypot"] = "cowrie"
cowrie_norm["protocol"] = "SSH"

cowrie_norm["source_ip"] = cowrie["src_ip"]
cowrie_norm["source_port"] = cowrie["src_port"]

cowrie_norm["destination_ip"] = cowrie["dest_ip"]
cowrie_norm["destination_port"] = cowrie["dest_port"]

cowrie_norm["session_id"] = cowrie["session"]

# ------------------------------------------
# Authentication
# ------------------------------------------

cowrie_norm["username"] = cowrie["username"]
cowrie_norm["password"] = cowrie["password"]

auth_events = cowrie["eventid"].isin([
    "cowrie.login.failed",
    "cowrie.login.success"
])

cowrie_norm["user_attempt"] = (
    cowrie["username"].notna() & auth_events
).astype(int)

cowrie_norm["pass_attempt"] = (
    cowrie["password"].notna() & auth_events
).astype(int)

cowrie_norm["auth_attempt"] = auth_events.astype(int)

# NULL username should not become <NA>
cowrie_norm["anonymous"] = (
    cowrie["username"]
    .fillna("")
    .astype(str)
    .str.lower()
    .eq("anonymous")
    .astype(int)
)

# ------------------------------------------
# Commands
# ------------------------------------------

cowrie_norm["command_count"] = (
    cowrie["eventid"]
    .eq("cowrie.command.input")
    .astype(int)
)

# ------------------------------------------
# GeoIP
# ------------------------------------------

cowrie_norm["country"] = cowrie["geoip.country_name"]
cowrie_norm["country_code"] = cowrie["geoip.country_code2"]
cowrie_norm["asn"] = cowrie["geoip.asn"]

cowrie_norm["ip_reputation"] = cowrie["ip_rep"]

# ------------------------------------------
# Attack / Intelligence fields
# ------------------------------------------

cowrie_norm["attack_type"] = np.nan
cowrie_norm["mitre_technique"] = np.nan
cowrie_norm["attack_stage"] = np.nan
cowrie_norm["severity"] = np.nan

# ------------------------------------------
# Dataset identification
# ------------------------------------------

cowrie_norm["source_dataset"] = "cowrie_ssh_preprocessed"

# ------------------------------------------
# Validation
# ------------------------------------------

print("Cowrie Normalized Shape:")
print(cowrie_norm.shape)

print("\nColumns:")
print(cowrie_norm.columns.tolist())

print("\nMissing Values:")
print(cowrie_norm.isna().sum())

print("\nFirst 5 Rows:")
display(cowrie_norm.head())

Cowrie Normalized Shape:
(33454, 24)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'command_count', 'country', 'country_code', 'asn', 'ip_reputation', 'attack_type', 'mitre_technique', 'attack_stage', 'severity', 'source_dataset']

Missing Values:
timestamp               0
honeypot                0
protocol                0
source_ip               0
source_port         26336
destination_ip      26326
destination_port    26326
session_id              0
username            27625
password            27638
user_attempt            0
pass_attempt            0
auth_attempt            0
anonymous               0
command_count           0
country                 1
country_code            1
asn                     1
ip_reputation       11285
attack_type         33454
mitre_technique     33454
attack_stage        33454
severity     

,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,username,password,...,command_count,country,country_code,asn,ip_reputation,attack_type,mitre_technique,attack_stage,severity,source_dataset
0,2025-06-11 13:21:58.483381+00:00,cowrie,SSH,8.138.44.199,56436.0,172.18.0.2,22.0,00009a53959a,NaN,NaN,...,0,China,CN,37963.0,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed
1,2025-06-11 13:21:58.483928+00:00,cowrie,SSH,8.138.44.199,NaN,NaN,NaN,00009a53959a,NaN,NaN,...,0,China,CN,37963.0,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed
2,2025-06-11 13:21:58.580759+00:00,cowrie,SSH,8.138.44.199,NaN,NaN,NaN,00009a53959a,NaN,NaN,...,0,China,CN,37963.0,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed
3,2025-06-11 13:21:58.874777+00:00,cowrie,SSH,8.138.44.199,NaN,NaN,NaN,00009a53959a,oracle,!QAZ@wsx,...,0,China,CN,37963.0,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed
4,2025-06-11 13:21:59.973329+00:00,cowrie,SSH,8.138.44.199,NaN,NaN,NaN,00009a53959a,NaN,NaN,...,0,China,CN,37963.0,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed


In [ ]:
print("========== COWRIE NORMALIZED VALIDATION ==========")

print("\nShape:")
print(cowrie_norm.shape)

print("\nDuplicates:")
print(cowrie_norm.duplicated().sum())

print("\nHoneypot Distribution:")
print(cowrie_norm["honeypot"].value_counts())

print("\nProtocol Distribution:")
print(cowrie_norm["protocol"].value_counts())

print("\nSource Dataset:")
print(cowrie_norm["source_dataset"].value_counts())

print("\nEvent-Level Authentication:")
print(cowrie_norm["auth_attempt"].value_counts())

print("\nCommand Events:")
print(cowrie_norm["command_count"].sum())

print("\nUnique Sessions:")
print(cowrie_norm["session_id"].nunique())

print("\nUnique Source IPs:")
print(cowrie_norm["source_ip"].nunique())

print("\nMissing Values:")
print(cowrie_norm.isna().sum())

========== COWRIE NORMALIZED VALIDATION ==========

Shape:
(33454, 24)

Duplicates:
0

Honeypot Distribution:
honeypot
cowrie    33454
Name: count, dtype: int64

Protocol Distribution:
protocol
SSH    33454
Name: count, dtype: int64

Source Dataset:
source_dataset
cowrie_ssh_preprocessed    33454
Name: count, dtype: int64

Event-Level Authentication:
auth_attempt
0    27625
1     5829
Name: count, dtype: int64

Command Events:
141

Unique Sessions:
7108

Unique Source IPs:
350

Missing Values:
timestamp               0
honeypot                0
protocol                0
source_ip               0
source_port         26336
destination_ip      26326
destination_port    26326
session_id              0
username            27625
password            27638
user_attempt            0
pass_attempt            0
auth_attempt            0
anonymous               0
command_count           0
country                 1
country_code            1
asn                     1
ip_reputation       11285
attack_

In [ ]:
# ==========================================
# TANNER HTTP NORMALIZATION
# ==========================================

import pandas as pd
import numpy as np

tanner_norm = pd.DataFrame()

# ------------------------------------------
# Core fields
# ------------------------------------------

tanner_norm["timestamp"] = pd.to_datetime(
    tanner["timestamp"],
    format="mixed",
    utc=True
)

tanner_norm["honeypot"] = "tanner"
tanner_norm["protocol"] = "HTTP"

# Source
tanner_norm["source_ip"] = tanner["peer.ip"]
tanner_norm["source_port"] = tanner["peer.port"]

# Destination
# Tanner dataset does not provide a clear destination IP field
tanner_norm["destination_ip"] = np.nan

# HTTP destination is represented by the service port
tanner_norm["destination_port"] = 80

# Session
tanner_norm["session_id"] = tanner["uuid"]

# ------------------------------------------
# Authentication
# ------------------------------------------

tanner_norm["username"] = np.nan
tanner_norm["password"] = np.nan

tanner_norm["user_attempt"] = 0
tanner_norm["pass_attempt"] = 0
tanner_norm["auth_attempt"] = 0
tanner_norm["anonymous"] = 0

# ------------------------------------------
# Commands
# ------------------------------------------

# HTTP does not have SSH/FTP command events
tanner_norm["command_count"] = 0

# ------------------------------------------
# GeoIP / Threat Intelligence
# ------------------------------------------

tanner_norm["country"] = np.nan
tanner_norm["country_code"] = np.nan
tanner_norm["asn"] = np.nan
tanner_norm["ip_reputation"] = np.nan

# ------------------------------------------
# Tanner attack enrichment
# ------------------------------------------

tanner_norm["attack_type"] = tanner["attack_type"]
tanner_norm["mitre_technique"] = tanner["mitre_technique"]
tanner_norm["attack_stage"] = tanner["attack_stage"]
tanner_norm["severity"] = tanner["severity"]

# ------------------------------------------
# Dataset identification
# ------------------------------------------

tanner_norm["source_dataset"] = "tanner_http_labeled"

# ------------------------------------------
# Inspect
# ------------------------------------------

print("Tanner Normalized Shape:")
print(tanner_norm.shape)

print("\nColumns:")
print(tanner_norm.columns.tolist())

print("\nFirst 5 Rows:")
display(tanner_norm.head())

print("\nMissing Values:")
print(tanner_norm.isna().sum())

Tanner Normalized Shape:
(2544, 24)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'command_count', 'country', 'country_code', 'asn', 'ip_reputation', 'attack_type', 'mitre_technique', 'attack_stage', 'severity', 'source_dataset']

First 5 Rows:


,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,username,password,...,command_count,country,country_code,asn,ip_reputation,attack_type,mitre_technique,attack_stage,severity,source_dataset
0,2020-01-22 04:56:10.963441+00:00,tanner,HTTP,45.243.162.246,33815,NaN,80,ade88c93-30a2-4cd0-902c-9b7f99df6579,NaN,NaN,...,0,NaN,NaN,NaN,NaN,Web Shell,T1505.003,Persistence,Critical,tanner_http_labeled
1,2020-01-22 04:56:11.888267+00:00,tanner,HTTP,45.243.162.246,33821,NaN,80,ade88c93-30a2-4cd0-902c-9b7f99df6579,NaN,NaN,...,0,NaN,NaN,NaN,NaN,Web Shell,T1505.003,Persistence,Critical,tanner_http_labeled
2,2020-01-22 05:41:42.580949+00:00,tanner,HTTP,187.0.174.26,58545,NaN,80,ade88c93-30a2-4cd0-902c-9b7f99df6579,NaN,NaN,...,0,NaN,NaN,NaN,NaN,Web Recon,NaN,NaN,NaN,tanner_http_labeled
3,2020-01-22 05:50:24.926195+00:00,tanner,HTTP,193.57.40.46,51714,NaN,80,ade88c93-30a2-4cd0-902c-9b7f99df6579,NaN,NaN,...,0,NaN,NaN,NaN,NaN,PHPUnit RCE,T1190,Initial Access,High,tanner_http_labeled
4,2020-01-22 06:21:34.802741+00:00,tanner,HTTP,169.197.108.42,37616,NaN,80,ade88c93-30a2-4cd0-902c-9b7f99df6579,NaN,NaN,...,0,NaN,NaN,NaN,NaN,Web Recon,NaN,NaN,NaN,tanner_http_labeled



Missing Values:
timestamp              0
honeypot               0
protocol               0
source_ip              0
source_port            0
destination_ip      2544
destination_port       0
session_id             0
username            2544
password            2544
user_attempt           0
pass_attempt           0
auth_attempt           0
anonymous              0
command_count          0
country             2544
country_code        2544
asn                 2544
ip_reputation       2544
attack_type            0
mitre_technique      902
attack_stage         902
severity             902
source_dataset         0
dtype: int64


In [ ]:
# ==========================================
# 1. Schema Alignment
# ==========================================

unified_schema = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity",
    "source_dataset"
]


print("Cowrie missing:")
print(set(unified_schema) - set(cowrie_norm.columns))

print("\nTanner missing:")
print(set(unified_schema) - set(tanner_norm.columns))

print("\nFTP missing:")
print(set(unified_schema) - set(ftp_unified.columns))

Cowrie missing:
set()

Tanner missing:
set()

FTP missing:
{'mitre_technique', 'severity', 'attack_type', 'attack_stage'}


In [ ]:
# ==========================================
# 2. Add missing schema fields to FTP
# ==========================================

ftp_final = ftp_unified.copy()

ftp_label_columns = [
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity"
]

for col in ftp_label_columns:
    if col not in ftp_final.columns:
        ftp_final[col] = pd.NA

print("FTP shape after schema alignment:")
print(ftp_final.shape)

print("\nFTP columns:")
print(ftp_final.columns.tolist())

FTP shape after schema alignment:
(1417, 24)

FTP columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'command_count', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'country', 'country_code', 'asn', 'ip_reputation', 'source_dataset', 'attack_type', 'mitre_technique', 'attack_stage', 'severity']


In [ ]:
# ==========================================
# 3. Force identical column order
# ==========================================

ftp_final = ftp_final.reindex(columns=unified_schema)
cowrie_final = cowrie_norm.reindex(columns=unified_schema)
tanner_final = tanner_norm.reindex(columns=unified_schema)

print("FTP :", ftp_final.shape)
print("Cowrie:", cowrie_final.shape)
print("Tanner:", tanner_final.shape)

print("\nSame schema?")
print(
    list(ftp_final.columns) ==
    list(cowrie_final.columns) ==
    list(tanner_final.columns)
)

FTP : (1417, 24)
Cowrie: (33454, 24)
Tanner: (2544, 24)

Same schema?
True


In [ ]:
# ==========================================
# 4. Merge all normalized honeypot datasets
# ==========================================

unified_security_events = pd.concat(
    [
        cowrie_final,
        tanner_final,
        ftp_final
    ],
    ignore_index=True
)

print("Unified Security Events Shape:")
print(unified_security_events.shape)

print("\nHoneypot Distribution:")
print(unified_security_events["honeypot"].value_counts())

print("\nProtocol Distribution:")
print(unified_security_events["protocol"].value_counts())

print("\nSource Dataset Distribution:")
print(unified_security_events["source_dataset"].value_counts())

Unified Security Events Shape:
(37415, 24)

Honeypot Distribution:
honeypot
cowrie       33454
tanner        2544
dionaea       1342
t-pot-ftp       75
Name: count, dtype: int64

Protocol Distribution:
protocol
SSH     33454
HTTP     2544
FTP      1417
Name: count, dtype: int64

Source Dataset Distribution:
source_dataset
cowrie_ssh_preprocessed    33454
tanner_http_labeled         2544
dionaea_ftp_clean           1342
ftp_clean_final               75
Name: count, dtype: int64


In [ ]:
# Save normalized unified dataset
unified_security_events.to_csv(
    "unified_security_events_normalized.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [ ]:
# ==========================================
# Unified Security Event Schema Validation
# ==========================================

import pandas as pd


df = unified_security_events.copy()

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# ------------------------------------------
# 1. Required Unified Schema
# ------------------------------------------

expected_schema = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "command_count",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity",
    "source_dataset"
]

# ------------------------------------------
# 2. Schema Check
# ------------------------------------------

missing_columns = set(expected_schema) - set(df.columns)
extra_columns = set(df.columns) - set(expected_schema)

print("\n========== SCHEMA VALIDATION ==========")

print("Missing columns:")
print(missing_columns if missing_columns else "None")

print("\nExtra columns:")
print(extra_columns if extra_columns else "None")

print("\nColumn order matches:")
print(df.columns.tolist() == expected_schema)

# ------------------------------------------
# 3. Data Type Check
# ------------------------------------------

print("\n========== DATA TYPES ==========")
print(df.dtypes)

# ------------------------------------------
# 4. Core Event Fields Validation
# ------------------------------------------

core_fields = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_dataset"
]

print("\n========== CORE FIELD NULL CHECK ==========")

for col in core_fields:
    print(f"{col}: {df[col].isna().sum()} missing")

# ------------------------------------------
# 5. Protocol / Honeypot Consistency
# ------------------------------------------

print("\n========== HONEYPOT / PROTOCOL ==========")

print("\nHoneypot distribution:")
print(df["honeypot"].value_counts(dropna=False))

print("\nProtocol distribution:")
print(df["protocol"].value_counts(dropna=False))

print("\nHoneypot × Protocol:")
print(pd.crosstab(df["honeypot"], df["protocol"]))

# ------------------------------------------
# 6. Source Dataset Consistency
# ------------------------------------------

print("\n========== SOURCE DATASET ==========")

print(df["source_dataset"].value_counts(dropna=False))

# ------------------------------------------
# 7. Duplicate Check
# ------------------------------------------

print("\n========== DUPLICATES ==========")
print("Duplicate rows:", df.duplicated().sum())

# ------------------------------------------
# 8. Timestamp Validation
# ------------------------------------------

print("\n========== TIMESTAMP ==========")

print("Min:", df["timestamp"].min())
print("Max:", df["timestamp"].max())

print("Invalid timestamps:",
      df["timestamp"].isna().sum())

# ------------------------------------------
# 9. Authentication Event Summary
# ------------------------------------------

print("\n========== AUTHENTICATION EVENTS ==========")

for col in [
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous"
]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

# ------------------------------------------
# 10. Attack Enrichment Coverage
# ------------------------------------------

print("\n========== ATTACK ENRICHMENT ==========")

for col in [
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity"
]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(20))

print("\n========================================")
print("Unified Security Event Schema validation completed.")
print("========================================")

Shape: (37415, 24)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'command_count', 'country', 'country_code', 'asn', 'ip_reputation', 'attack_type', 'mitre_technique', 'attack_stage', 'severity', 'source_dataset']

========== SCHEMA VALIDATION ==========
Missing columns:
None

Extra columns:
None

Column order matches:
False

========== DATA TYPES ==========
timestamp           datetime64[ns, UTC]
honeypot                         object
protocol                         object
source_ip                        object
source_port                     float64
destination_ip                   object
destination_port                float64
session_id                       object
username                         object
password                         object
user_attempt                    float64
pass_attempt                    f

In [ ]:
# ==========================================
# SAVE UNIFIED BASELINE DATASET
# ==========================================

unified_security_events.to_csv(
    "unified_security_events_baseline.csv",
    index=False
)

print("Saved successfully.")
print("Shape:", unified_security_events.shape)
print("File: unified_security_events_baseline.csv")

Saved successfully.
Shape: (37415, 24)
File: unified_security_events_baseline.csv


In [ ]:
# Reload and verify
baseline_check = pd.read_csv(
    "unified_security_events_baseline.csv",
    low_memory=False
)

print("Reloaded shape:", baseline_check.shape)
print("Columns:", len(baseline_check.columns))
print("Duplicates:", baseline_check.duplicated().sum())
print("Missing core fields:")

core_fields = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_dataset"
]

print(baseline_check[core_fields].isna().sum())

Reloaded shape: (37415, 24)
Columns: 24
Duplicates: 0
Missing core fields:
timestamp         0
honeypot          0
protocol          0
source_ip         0
source_dataset    0
dtype: int64


In [ ]:
%whos DataFrame

Variable                  Type         Data/Info
------------------------------------------------
baseline_check            DataFrame                             <...>[37415 rows x 24 columns]
cowrie                    DataFrame            t-pot_hostname   <...>[33454 rows x 73 columns]
cowrie_final              DataFrame                             <...>[33454 rows x 24 columns]
cowrie_norm               DataFrame                             <...>[33454 rows x 24 columns]
df                        DataFrame                             <...>[37415 rows x 24 columns]
dionaea                   DataFrame          connection connecti<...>n[1344 rows x 13 columns]
dionaea_norm              DataFrame                             <...>n[1344 rows x 20 columns]
dup_check                 DataFrame                             <...> NaN    ftp_clean_final  
duplicates                DataFrame                             <...> NaN    ftp_clean_final  
ftp1                      DataFrame            

In [ ]:
print("Cowrie raw:", cowrie.shape)
print("Cowrie normalized:", cowrie_norm.shape)
print("Cowrie final:", cowrie_final.shape)

print("\nUnified current:", unified_security_events.shape)

print("\nExpected unified:")
print(54934 + 2544 + 1417)

Cowrie raw: (33454, 73)
Cowrie normalized: (33454, 24)
Cowrie final: (33454, 24)

Unified current: (37415, 24)

Expected unified:
58895


In [ ]:
print("Cowrie final source dataset:")
print(cowrie_final["source_dataset"].value_counts(dropna=False))

print("\nCowrie final duplicates:")
print(cowrie_final.duplicated().sum())

print("\nCowrie final sessions:")
print(cowrie_final["session_id"].nunique())

print("\nCowrie normalized sessions:")
print(cowrie_norm["session_id"].nunique())

Cowrie final source dataset:
source_dataset
cowrie_ssh_preprocessed    33454
Name: count, dtype: int64

Cowrie final duplicates:
0

Cowrie final sessions:
7108

Cowrie normalized sessions:
7108


In [ ]:
import os

print(os.listdir())

['.config', 'cowrie_ssh_preprocessed.csv', 'ftp_clean_final.csv', 'unified_security_events_normalized.csv', 'tanner_http_labeled.csv', 'dionaea_ftp_clean.csv', 'unified_security_events_baseline.csv', 'sample_data']


In [ ]:
import pandas as pd

cowrie_check = pd.read_csv(
    "cowrie_ssh_preprocessed.csv",
    low_memory=False
)

print("Shape:", cowrie_check.shape)
print("Columns:", len(cowrie_check.columns))
print("Duplicates:", cowrie_check.duplicated().sum())

Shape: (54934, 73)
Columns: 73
Duplicates: 0


In [ ]:
import pandas as pd

# Load the correct Cowrie dataset
cowrie = pd.read_csv(
    "cowrie_ssh_preprocessed.csv",
    low_memory=False
)

# Load the other normalized datasets from source files
ftp1 = pd.read_csv("ftp_clean_final.csv", low_memory=False)
dionaea = pd.read_csv("dionaea_ftp_clean.csv", low_memory=False)
tanner = pd.read_csv("tanner_http_labeled.csv", low_memory=False)

print("Cowrie:", cowrie.shape)
print("FTP 1:", ftp1.shape)
print("Dionaea:", dionaea.shape)
print("Tanner:", tanner.shape)

Cowrie: (54934, 73)
FTP 1: (76, 36)
Dionaea: (1344, 13)
Tanner: (2544, 62)


In [ ]:
import pandas as pd
import os

# =========================
# LOAD RAW DATA
# =========================

cowrie = pd.read_csv(
    "cowrie_ssh_preprocessed.csv",
    low_memory=False
)

ftp1 = pd.read_csv(
    "ftp_clean_final.csv",
    low_memory=False
)

dionaea = pd.read_csv(
    "dionaea_ftp_clean.csv",
    low_memory=False
)

tanner = pd.read_csv(
    "tanner_http_labeled.csv",
    low_memory=False
)

# =========================
# RAW DATA SUMMARY
# =========================

raw_datasets = {
    "cowrie": cowrie,
    "ftp1": ftp1,
    "dionaea": dionaea,
    "tanner": tanner
}

print("=" * 60)
print("RAW DATA BASELINE")
print("=" * 60)

for name, data in raw_datasets.items():
    print(f"\n{name.upper()}")
    print("-" * 40)
    print("Shape:", data.shape)
    print("Columns:", len(data.columns))
    print("Duplicates:", data.duplicated().sum())
    print("Missing cells:", data.isna().sum().sum())

RAW DATA BASELINE

COWRIE
----------------------------------------
Shape: (54934, 73)
Columns: 73
Duplicates: 0
Missing cells: 1402955

FTP1
----------------------------------------
Shape: (76, 36)
Columns: 36
Duplicates: 0
Missing cells: 181

DIONAEA
----------------------------------------
Shape: (1344, 13)
Columns: 13
Duplicates: 0
Missing cells: 47

TANNER
----------------------------------------
Shape: (2544, 62)
Columns: 62
Duplicates: 0
Missing cells: 102874


In [ ]:
# =========================
# CREATE BACKUP DIRECTORY
# =========================

os.makedirs("pipeline_backup/raw", exist_ok=True)

cowrie.to_csv(
    "pipeline_backup/raw/cowrie_raw_backup.csv",
    index=False
)

ftp1.to_csv(
    "pipeline_backup/raw/ftp1_raw_backup.csv",
    index=False
)

dionaea.to_csv(
    "pipeline_backup/raw/dionaea_raw_backup.csv",
    index=False
)

tanner.to_csv(
    "pipeline_backup/raw/tanner_raw_backup.csv",
    index=False
)

print("Raw backup completed.")

Raw backup completed.


In [ ]:
baseline = pd.read_csv(
    "unified_security_events_baseline.csv",
    low_memory=False
)

print("Shape:", baseline.shape)
print("\nColumns:")
print(baseline.columns.tolist())

print("\nDuplicates:")
print(baseline.duplicated().sum())

print("\nHoneypot Distribution:")
print(baseline["honeypot"].value_counts(dropna=False))

print("\nProtocol Distribution:")
print(baseline["protocol"].value_counts(dropna=False))

print("\nSource Dataset:")
print(baseline["source_dataset"].value_counts(dropna=False))

print("\nCore Missing Values:")
print(
    baseline[
        ["timestamp", "honeypot", "protocol",
         "source_ip", "source_dataset"]
    ].isna().sum()
)

Shape: (37415, 24)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'command_count', 'country', 'country_code', 'asn', 'ip_reputation', 'attack_type', 'mitre_technique', 'attack_stage', 'severity', 'source_dataset']

Duplicates:
0

Honeypot Distribution:
honeypot
cowrie       33454
tanner        2544
dionaea       1342
t-pot-ftp       75
Name: count, dtype: int64

Protocol Distribution:
protocol
SSH     33454
HTTP     2544
FTP      1417
Name: count, dtype: int64

Source Dataset:
source_dataset
cowrie_ssh_preprocessed    33454
tanner_http_labeled         2544
dionaea_ftp_clean           1342
ftp_clean_final               75
Name: count, dtype: int64

Core Missing Values:
timestamp         0
honeypot          0
protocol          0
source_ip         0
source_dataset    0
dtype: int64


In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# STEP 1 — COWRIE RAW → UNIFIED NORMALIZED SCHEMA
# ============================================================

# Load raw Cowrie
cowrie = pd.read_csv(
    "cowrie_ssh_preprocessed.csv",
    low_memory=False
)

print("Cowrie Raw Shape:", cowrie.shape)

# ------------------------------------------------------------
# Helper: safely get a column
# ------------------------------------------------------------
def get_col(df, col, default=np.nan):
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index)


# ------------------------------------------------------------
# Create normalized dataframe
# ------------------------------------------------------------
cowrie_norm = pd.DataFrame(index=cowrie.index)

# Core identity
cowrie_norm["timestamp"] = pd.to_datetime(
    get_col(cowrie, "timestamp"),
    errors="coerce",
    utc=True
)

cowrie_norm["honeypot"] = "cowrie"
cowrie_norm["protocol"] = "SSH"

# Network
cowrie_norm["source_ip"] = get_col(cowrie, "src_ip")
cowrie_norm["source_port"] = pd.to_numeric(
    get_col(cowrie, "src_port"),
    errors="coerce"
)

cowrie_norm["destination_ip"] = get_col(cowrie, "dest_ip")
cowrie_norm["destination_port"] = pd.to_numeric(
    get_col(cowrie, "dest_port"),
    errors="coerce"
)

# Session
cowrie_norm["session_id"] = get_col(cowrie, "session")

# Authentication
cowrie_norm["username"] = get_col(cowrie, "username")
cowrie_norm["password"] = get_col(cowrie, "password")

# ------------------------------------------------------------
# Authentication indicators
# ------------------------------------------------------------

cowrie_norm["user_attempt"] = (
    cowrie_norm["username"]
    .notna()
    .astype(int)
)

cowrie_norm["pass_attempt"] = (
    cowrie_norm["password"]
    .notna()
    .astype(int)
)

cowrie_norm["auth_attempt"] = (
    (
        cowrie_norm["username"].notna()
        |
        cowrie_norm["password"].notna()
    )
    .astype(int)
)

cowrie_norm["anonymous"] = (
    cowrie_norm["username"]
    .astype("string")
    .str.lower()
    .eq("anonymous")
    .fillna(False)
    .astype(int)
)

# ------------------------------------------------------------
# Command events
# ------------------------------------------------------------

# Cowrie command/input events are represented through "input"
cowrie_norm["command_count"] = (
    get_col(cowrie, "input")
    .notna()
    .astype(int)
)

# ------------------------------------------------------------
# GeoIP
# ------------------------------------------------------------

cowrie_norm["country"] = get_col(
    cowrie, "geoip_ext.country_name"
)

cowrie_norm["country_code"] = get_col(
    cowrie, "geoip_ext.country_code2"
)

cowrie_norm["asn"] = pd.to_numeric(
    get_col(cowrie, "geoip_ext.asn"),
    errors="coerce"
)

# ------------------------------------------------------------
# Threat intelligence
# ------------------------------------------------------------

cowrie_norm["ip_reputation"] = get_col(
    cowrie, "ip_rep"
)

# Cowrie raw dataset does not currently contain the same
# labeled attack enrichment as Tanner.
cowrie_norm["attack_type"] = np.nan
cowrie_norm["mitre_technique"] = np.nan
cowrie_norm["attack_stage"] = np.nan
cowrie_norm["severity"] = np.nan

# ------------------------------------------------------------
# Dataset provenance
# ------------------------------------------------------------

cowrie_norm["source_dataset"] = (
    "cowrie_ssh_preprocessed"
)

# ------------------------------------------------------------
# Final schema
# ------------------------------------------------------------

UNIFIED_SCHEMA = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity",
    "source_dataset"
]

cowrie_norm = cowrie_norm[UNIFIED_SCHEMA]

print("\n" + "=" * 60)
print("COWRIE NORMALIZED")
print("=" * 60)

print("Shape:", cowrie_norm.shape)
print("Columns:", len(cowrie_norm.columns))

print("\nMissing columns:")
print(set(UNIFIED_SCHEMA) - set(cowrie_norm.columns))

print("\nDuplicates:", cowrie_norm.duplicated().sum())

print("\nCore missing values:")
print(
    cowrie_norm[
        [
            "timestamp",
            "honeypot",
            "protocol",
            "source_ip",
            "source_dataset"
        ]
    ].isna().sum()
)

print("\nAuthentication:")
print(
    cowrie_norm[
        [
            "user_attempt",
            "pass_attempt",
            "auth_attempt",
            "anonymous"
        ]
    ].sum()
)

print("\nCommand events:")
print(
    cowrie_norm["command_count"].sum()
)

print("\nUnique sessions:")
print(
    cowrie_norm["session_id"].nunique()
)

print("\nUnique source IPs:")
print(
    cowrie_norm["source_ip"].nunique()
)

print("\nFirst 5 rows:")
display(cowrie_norm.head())

Cowrie Raw Shape: (54934, 73)

COWRIE NORMALIZED
Shape: (54934, 24)
Columns: 24

Missing columns:
set()

Duplicates: 0

Core missing values:
timestamp         0
honeypot          0
protocol          0
source_ip         0
source_dataset    0
dtype: int64

Authentication:
user_attempt    9592
pass_attempt    9575
auth_attempt    9592
anonymous          0
dtype: int64

Command events:
264

Unique sessions:
11663

Unique source IPs:
436

First 5 rows:


,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,username,password,...,command_count,country,country_code,asn,ip_reputation,attack_type,mitre_technique,attack_stage,severity,source_dataset
0,2025-06-11 13:21:58.483381+00:00,cowrie,SSH,8.138.44.199,56436.0,172.18.0.2,22.0,00009a53959a,NaN,NaN,...,0,India,IN,8075,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed
1,2025-06-11 13:21:58.483928+00:00,cowrie,SSH,8.138.44.199,NaN,NaN,NaN,00009a53959a,NaN,NaN,...,0,India,IN,8075,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed
2,2025-06-11 13:21:58.580759+00:00,cowrie,SSH,8.138.44.199,NaN,NaN,NaN,00009a53959a,NaN,NaN,...,0,India,IN,8075,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed
3,2025-06-11 13:21:58.874777+00:00,cowrie,SSH,8.138.44.199,NaN,NaN,NaN,00009a53959a,oracle,!QAZ@wsx,...,0,India,IN,8075,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed
4,2025-06-11 13:21:59.973329+00:00,cowrie,SSH,8.138.44.199,NaN,NaN,NaN,00009a53959a,NaN,NaN,...,0,India,IN,8075,known attacker,NaN,NaN,NaN,NaN,cowrie_ssh_preprocessed


In [ ]:
UNIFIED_COLUMNS = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity",
    "source_dataset"
]

In [ ]:
import pandas as pd
import numpy as np

ftp1 = pd.read_csv("ftp_clean_final.csv")
dionaea = pd.read_csv("dionaea_ftp_clean.csv")

print("FTP1:", ftp1.shape)
print("Dionaea:", dionaea.shape)

FTP1: (76, 36)
Dionaea: (1344, 13)


In [ ]:
print(dionaea.columns.tolist())

['connection', 'connection_protocol', 'connection_timestamp', 'local_port', 'remote_host', 'remote_port', 'timestamp', 'date', 'login', 'login_username', 'login_password', 'hour', 'weekday']


In [ ]:
print(dionaea.head())

   connection connection_protocol  connection_timestamp  local_port  \
0         639                ftpd          1.581746e+09          21   
1         639                ftpd          1.581746e+09          21   
2         639                ftpd          1.581746e+09          21   
3       20231                ftpd          1.581785e+09          21   
4       20231                ftpd          1.581785e+09          21   

       remote_host  remote_port                      timestamp        date  \
0  192.241.231.243        56190  2020-02-15 05:48:27.558816671  2020-02-15   
1  192.241.231.243        56190  2020-02-15 05:48:27.558816671  2020-02-15   
2  192.241.231.243        56190  2020-02-15 05:48:27.558816671  2020-02-15   
3   104.206.128.30        63509  2020-02-15 16:36:56.719931841  2020-02-15   
4   104.206.128.30        63509  2020-02-15 16:36:56.719931841  2020-02-15   

     login login_username login_password  hour   weekday  
0    381.0             sa       dddddddd     

In [ ]:
import pandas as pd

dionaea_norm = pd.DataFrame()

# =========================
# Core fields
# =========================

dionaea_norm["timestamp"] = pd.to_datetime(
    dionaea["timestamp"], utc=True, errors="coerce"
)

dionaea_norm["honeypot"] = "dionaea"
dionaea_norm["protocol"] = "FTP"

dionaea_norm["source_ip"] = dionaea["remote_host"]
dionaea_norm["source_port"] = pd.to_numeric(
    dionaea["remote_port"], errors="coerce"
)

dionaea_norm["destination_ip"] = pd.NA

dionaea_norm["destination_port"] = pd.to_numeric(
    dionaea["local_port"], errors="coerce"
)

dionaea_norm["session_id"] = dionaea["connection"].astype("string")

# =========================
# Authentication fields
# =========================

dionaea_norm["username"] = dionaea["login_username"]
dionaea_norm["password"] = dionaea["login_password"]

# Dionaea rows represent login activity
dionaea_norm["user_attempt"] = (
    dionaea["login_username"].notna()
).astype(int)

dionaea_norm["pass_attempt"] = (
    dionaea["login_password"].notna()
).astype(int)

dionaea_norm["auth_attempt"] = (
    dionaea["login"].notna()
).astype(int)

dionaea_norm["anonymous"] = (
    dionaea["login_username"]
    .astype("string")
    .str.lower()
    .eq("anonymous")
    .fillna(False)
    .astype(int)
)

# =========================
# Command / enrichment fields
# =========================

dionaea_norm["command_count"] = 0

dionaea_norm["country"] = pd.NA
dionaea_norm["country_code"] = pd.NA
dionaea_norm["asn"] = pd.NA
dionaea_norm["ip_reputation"] = pd.NA

# =========================
# Attack enrichment
# =========================

dionaea_norm["attack_type"] = pd.NA
dionaea_norm["mitre_technique"] = pd.NA
dionaea_norm["attack_stage"] = pd.NA
dionaea_norm["severity"] = pd.NA

# =========================
# Dataset identifier
# =========================

dionaea_norm["source_dataset"] = "dionaea_ftp_clean"

# =========================
# Reorder to unified schema
# =========================

unified_schema = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity",
    "source_dataset"
]

dionaea_norm = dionaea_norm[unified_schema]

print("Dionaea Normalized Shape:", dionaea_norm.shape)
print("\nColumns:")
print(dionaea_norm.columns.tolist())

print("\nMissing Core Fields:")
print(
    dionaea_norm[
        ["timestamp", "honeypot", "protocol", "source_ip", "source_dataset"]
    ].isna().sum()
)

print("\nDuplicates:", dionaea_norm.duplicated().sum())

print("\nFirst 5 Rows:")
display(dionaea_norm.head())

Dionaea Normalized Shape: (1344, 24)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'command_count', 'country', 'country_code', 'asn', 'ip_reputation', 'attack_type', 'mitre_technique', 'attack_stage', 'severity', 'source_dataset']

Missing Core Fields:
timestamp         0
honeypot          0
protocol          0
source_ip         0
source_dataset    0
dtype: int64

Duplicates: 2

First 5 Rows:


,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,username,password,...,command_count,country,country_code,asn,ip_reputation,attack_type,mitre_technique,attack_stage,severity,source_dataset
0,2020-02-15 05:48:27.558816671+00:00,dionaea,FTP,192.241.231.243,56190,<NA>,21,639,sa,dddddddd,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean
1,2020-02-15 05:48:27.558816671+00:00,dionaea,FTP,192.241.231.243,56190,<NA>,21,639,sa,1qazxsw2,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean
2,2020-02-15 05:48:27.558816671+00:00,dionaea,FTP,192.241.231.243,56190,<NA>,21,639,sa,0123456789,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean
3,2020-02-15 16:36:56.719931841+00:00,dionaea,FTP,104.206.128.30,63509,<NA>,21,20231,sa,feifei521,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean
4,2020-02-15 16:36:56.719931841+00:00,dionaea,FTP,104.206.128.30,63509,<NA>,21,20231,sa,19890125,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean


In [ ]:
dionaea_duplicates = dionaea_norm[
    dionaea_norm.duplicated(keep=False)
].sort_values(
    by=["timestamp", "source_ip", "session_id", "username", "password"]
)

print("Number of duplicate rows:", len(dionaea_duplicates))
display(dionaea_duplicates)

Number of duplicate rows: 4


,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,username,password,...,command_count,country,country_code,asn,ip_reputation,attack_type,mitre_technique,attack_stage,severity,source_dataset
658,2020-01-26 08:18:57.813625336+00:00,dionaea,FTP,80.82.70.118,60000,<NA>,21,14391,sa,19840517,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean
660,2020-01-26 08:18:57.813625336+00:00,dionaea,FTP,80.82.70.118,60000,<NA>,21,14391,sa,19840517,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean
44,2020-02-14 09:06:40.307245255+00:00,dionaea,FTP,71.6.232.4,33694,<NA>,21,1763,sa,19881020,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean
45,2020-02-14 09:06:40.307245255+00:00,dionaea,FTP,71.6.232.4,33694,<NA>,21,1763,sa,19881020,...,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,dionaea_ftp_clean


In [ ]:
# Remove exact duplicate rows only
dionaea_norm = (
    dionaea_norm
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

print("Dionaea shape after exact duplicate removal:", dionaea_norm.shape)
print("Remaining duplicates:", dionaea_norm.duplicated().sum())

print("\nSource dataset:")
print(dionaea_norm["source_dataset"].value_counts())

Dionaea shape after exact duplicate removal: (1342, 24)
Remaining duplicates: 0

Source dataset:
source_dataset
dionaea_ftp_clean    1342
Name: count, dtype: int64


In [ ]:
print(ftp1.columns.tolist())
print("\nFirst 5 rows:")
display(ftp1.head())

['t-pot_hostname', 't-pot_ip_int', 't-pot_ip_ext', 'src_port', '@timestamp', 'host', 'dest_ip', 'src_ip', 'geoip_ext.timezone', 'geoip_ext.latitude', 'geoip_ext.postal_code', 'geoip_ext.country_code2', 'geoip_ext.country_name', 'geoip_ext.country_code3', 'geoip_ext.region_code', 'geoip_ext.continent_code', 'geoip_ext.region_name', 'geoip_ext.city_name', 'geoip_ext.longitude', 'geoip_ext.location.lat', 'geoip_ext.location.lon', 'geoip_ext.ip', 'headers.content_length', 'geoip.country_code2', 'geoip.region_code', 'geoip.asn', 'geoip_ext.dma_code', 'geoip.city_name', 'ip_rep', 'command_count', 'has_USER', 'has_PASS', 'has_AUTH', 'has_username', 'has_password', 'is_anonymous']

First 5 rows:


,t-pot_hostname,t-pot_ip_int,t-pot_ip_ext,src_port,@timestamp,host,dest_ip,src_ip,geoip_ext.timezone,geoip_ext.latitude,...,geoip_ext.dma_code,geoip.city_name,ip_rep,command_count,has_USER,has_PASS,has_AUTH,has_username,has_password,is_anonymous
0,vm-southafricanorth,10.4.0.4,4.221.107.209,51168,2025-06-11T15:33:07.648Z,172.18.0.6,172.21.0.2,198.235.24.69,Africa/Johannesburg,-26.2309,...,NaN,NaN,NaN,1,0,0,1,0,0,0
1,vm-centralus,10.2.0.4,52.185.37.223,37444,2025-06-11T15:27:12.437Z,172.18.0.6,172.20.0.2,106.75.50.248,America/Chicago,41.6015,...,679.0,NaN,NaN,1,0,0,1,0,0,0
2,vm-centralus,10.2.0.4,52.185.37.223,36978,2025-06-11T15:27:11.970Z,172.18.0.6,172.20.0.2,106.75.50.248,America/Chicago,41.6015,...,679.0,NaN,NaN,1,1,0,0,0,0,0
3,vm-centralus,10.2.0.4,52.185.37.223,53820,2025-06-11T15:10:03.296Z,172.18.0.6,172.20.0.2,20.38.33.1,America/Chicago,41.6015,...,679.0,Phoenix,NaN,2,0,0,1,0,0,0
4,vm-southafricanorth,10.4.0.4,4.221.107.209,38820,2025-06-11T15:00:35.116Z,172.18.0.6,172.21.0.2,20.65.193.113,Africa/Johannesburg,-26.2309,...,NaN,San Antonio,known attacker,2,0,0,1,0,0,0


In [ ]:
import pandas as pd

ftp1_norm = pd.DataFrame()

# ==========================================
# Core event fields
# ==========================================

ftp1_norm["timestamp"] = pd.to_datetime(
    ftp1["@timestamp"],
    utc=True,
    errors="coerce"
)

ftp1_norm["honeypot"] = "t-pot-ftp"
ftp1_norm["protocol"] = "FTP"

ftp1_norm["source_ip"] = ftp1["src_ip"]

ftp1_norm["source_port"] = pd.to_numeric(
    ftp1["src_port"],
    errors="coerce"
)

ftp1_norm["destination_ip"] = ftp1["dest_ip"]

# T-Pot FTP logs don't provide a destination port
# in this preprocessed dataset
ftp1_norm["destination_port"] = pd.NA

ftp1_norm["session_id"] = pd.NA

# ==========================================
# Authentication fields
# ==========================================

ftp1_norm["username"] = pd.NA
ftp1_norm["password"] = pd.NA

ftp1_norm["user_attempt"] = pd.to_numeric(
    ftp1["has_USER"],
    errors="coerce"
).fillna(0).astype(int)

ftp1_norm["pass_attempt"] = pd.to_numeric(
    ftp1["has_PASS"],
    errors="coerce"
).fillna(0).astype(int)

ftp1_norm["auth_attempt"] = pd.to_numeric(
    ftp1["has_AUTH"],
    errors="coerce"
).fillna(0).astype(int)

ftp1_norm["anonymous"] = pd.to_numeric(
    ftp1["is_anonymous"],
    errors="coerce"
).fillna(0).astype(int)

# ==========================================
# Command information
# ==========================================

ftp1_norm["command_count"] = pd.to_numeric(
    ftp1["command_count"],
    errors="coerce"
).fillna(0)

# ==========================================
# GeoIP / Threat Intelligence fields
# ==========================================

ftp1_norm["country"] = ftp1["geoip_ext.country_name"]

ftp1_norm["country_code"] = ftp1["geoip_ext.country_code2"]

ftp1_norm["asn"] = pd.to_numeric(
    ftp1["geoip.asn"],
    errors="coerce"
)

ftp1_norm["ip_reputation"] = ftp1["ip_rep"]

# ==========================================
# Attack enrichment
# ==========================================

ftp1_norm["attack_type"] = pd.NA
ftp1_norm["mitre_technique"] = pd.NA
ftp1_norm["attack_stage"] = pd.NA
ftp1_norm["severity"] = pd.NA

# ==========================================
# Dataset identifier
# ==========================================

ftp1_norm["source_dataset"] = "ftp_clean_final"

# ==========================================
# Unified schema
# ==========================================

unified_schema = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity",
    "source_dataset"
]

ftp1_norm = ftp1_norm[unified_schema]

print("FTP1 Normalized Shape:", ftp1_norm.shape)

print("\nColumns:")
print(ftp1_norm.columns.tolist())

print("\nMissing Core Fields:")
print(
    ftp1_norm[
        [
            "timestamp",
            "honeypot",
            "protocol",
            "source_ip",
            "source_dataset"
        ]
    ].isna().sum()
)

print("\nDuplicates:", ftp1_norm.duplicated().sum())

print("\nAuthentication:")
print(
    ftp1_norm[
        [
            "user_attempt",
            "pass_attempt",
            "auth_attempt",
            "anonymous"
        ]
    ].sum()
)

print("\nFirst 5 Rows:")
display(ftp1_norm.head())

FTP1 Normalized Shape: (76, 24)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'command_count', 'country', 'country_code', 'asn', 'ip_reputation', 'attack_type', 'mitre_technique', 'attack_stage', 'severity', 'source_dataset']

Missing Core Fields:
timestamp         0
honeypot          0
protocol          0
source_ip         0
source_dataset    0
dtype: int64

Duplicates: 1

Authentication:
user_attempt    11
pass_attempt     7
auth_attempt    29
anonymous        4
dtype: int64

First 5 Rows:


,timestamp,honeypot,protocol,source_ip,source_port,destination_ip,destination_port,session_id,username,password,...,command_count,country,country_code,asn,ip_reputation,attack_type,mitre_technique,attack_stage,severity,source_dataset
0,2025-06-11 15:33:07.648000+00:00,t-pot-ftp,FTP,198.235.24.69,51168,172.21.0.2,<NA>,<NA>,<NA>,<NA>,...,1,South Africa,ZA,396982,NaN,<NA>,<NA>,<NA>,<NA>,ftp_clean_final
1,2025-06-11 15:27:12.437000+00:00,t-pot-ftp,FTP,106.75.50.248,37444,172.20.0.2,<NA>,<NA>,<NA>,<NA>,...,1,United States,US,4808,NaN,<NA>,<NA>,<NA>,<NA>,ftp_clean_final
2,2025-06-11 15:27:11.970000+00:00,t-pot-ftp,FTP,106.75.50.248,36978,172.20.0.2,<NA>,<NA>,<NA>,<NA>,...,1,United States,US,4808,NaN,<NA>,<NA>,<NA>,<NA>,ftp_clean_final
3,2025-06-11 15:10:03.296000+00:00,t-pot-ftp,FTP,20.38.33.1,53820,172.20.0.2,<NA>,<NA>,<NA>,<NA>,...,2,United States,US,8075,NaN,<NA>,<NA>,<NA>,<NA>,ftp_clean_final
4,2025-06-11 15:00:35.116000+00:00,t-pot-ftp,FTP,20.65.193.113,38820,172.21.0.2,<NA>,<NA>,<NA>,<NA>,...,2,South Africa,ZA,8075,known attacker,<NA>,<NA>,<NA>,<NA>,ftp_clean_final


In [ ]:
# Remove exact duplicates from FTP1
ftp1_norm = (
    ftp1_norm
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

print("FTP1 final shape:", ftp1_norm.shape)
print("Remaining duplicates:", ftp1_norm.duplicated().sum())

print("\nSource dataset:")
print(ftp1_norm["source_dataset"].value_counts())

FTP1 final shape: (75, 24)
Remaining duplicates: 0

Source dataset:
source_dataset
ftp_clean_final    75
Name: count, dtype: int64


In [ ]:
# ==========================================
# UNIFIED SECURITY EVENTS
# Schema Alignment + Merge
# ==========================================

unified_schema = [
    "timestamp",
    "honeypot",
    "protocol",
    "source_ip",
    "source_port",
    "destination_ip",
    "destination_port",
    "session_id",
    "username",
    "password",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "country",
    "country_code",
    "asn",
    "ip_reputation",
    "attack_type",
    "mitre_technique",
    "attack_stage",
    "severity",
    "source_dataset"
]

# Make sure every dataset follows exactly the same schema
cowrie_final = cowrie_norm[unified_schema].copy()
tanner_final = tanner_norm[unified_schema].copy()
dionaea_final = dionaea_norm[unified_schema].copy()
ftp_final = ftp1_norm[unified_schema].copy()

# Merge
unified_security_events = pd.concat(
    [
        cowrie_final,
        tanner_final,
        dionaea_final,
        ftp_final
    ],
    ignore_index=True
)

print("Unified Security Events Shape:")
print(unified_security_events.shape)

print("\nColumns:")
print(unified_security_events.columns.tolist())

print("\nDuplicates:")
print(unified_security_events.duplicated().sum())

print("\nHoneypot Distribution:")
print(unified_security_events["honeypot"].value_counts())

print("\nProtocol Distribution:")
print(unified_security_events["protocol"].value_counts())

print("\nSource Dataset Distribution:")
print(unified_security_events["source_dataset"].value_counts())

/tmp/ipykernel_1475/1650641377.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unified_security_events = pd.concat(


Unified Security Events Shape:
(58895, 24)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'command_count', 'country', 'country_code', 'asn', 'ip_reputation', 'attack_type', 'mitre_technique', 'attack_stage', 'severity', 'source_dataset']

Duplicates:
0

Honeypot Distribution:
honeypot
cowrie       54934
tanner        2544
dionaea       1342
t-pot-ftp       75
Name: count, dtype: int64

Protocol Distribution:
protocol
SSH     54934
HTTP     2544
FTP      1417
Name: count, dtype: int64

Source Dataset Distribution:
source_dataset
cowrie_ssh_preprocessed    54934
tanner_http_labeled         2544
dionaea_ftp_clean           1342
ftp_clean_final               75
Name: count, dtype: int64


In [ ]:
# ==========================================
# FINAL UNIFIED NORMALIZED BASELINE
# ==========================================

print("=" * 60)
print("FINAL UNIFIED NORMALIZED BASELINE VALIDATION")
print("=" * 60)

print("\nShape:")
print(unified_security_events.shape)

print("\nColumns:")
print(unified_security_events.columns.tolist())

print("\nMissing Core Fields:")
print(
    unified_security_events[
        [
            "timestamp",
            "honeypot",
            "protocol",
            "source_ip",
            "source_dataset"
        ]
    ].isna().sum()
)

print("\nDuplicates:")
print(unified_security_events.duplicated().sum())

print("\nHoneypot Distribution:")
print(unified_security_events["honeypot"].value_counts())

print("\nProtocol Distribution:")
print(unified_security_events["protocol"].value_counts())

print("\nSource Dataset Distribution:")
print(unified_security_events["source_dataset"].value_counts())

print("\nTimestamp Range:")
print("Min:", unified_security_events["timestamp"].min())
print("Max:", unified_security_events["timestamp"].max())

print("\nSchema column count:")
print(len(unified_security_events.columns))

FINAL UNIFIED NORMALIZED BASELINE VALIDATION

Shape:
(58895, 24)

Columns:
['timestamp', 'honeypot', 'protocol', 'source_ip', 'source_port', 'destination_ip', 'destination_port', 'session_id', 'username', 'password', 'user_attempt', 'pass_attempt', 'auth_attempt', 'anonymous', 'command_count', 'country', 'country_code', 'asn', 'ip_reputation', 'attack_type', 'mitre_technique', 'attack_stage', 'severity', 'source_dataset']

Missing Core Fields:
timestamp         0
honeypot          0
protocol          0
source_ip         0
source_dataset    0
dtype: int64

Duplicates:
0

Honeypot Distribution:
honeypot
cowrie       54934
tanner        2544
dionaea       1342
t-pot-ftp       75
Name: count, dtype: int64

Protocol Distribution:
protocol
SSH     54934
HTTP     2544
FTP      1417
Name: count, dtype: int64

Source Dataset Distribution:
source_dataset
cowrie_ssh_preprocessed    54934
tanner_http_labeled         2544
dionaea_ftp_clean           1342
ftp_clean_final               75
Name: count

In [ ]:
unified_security_events.to_csv(
    "unified_security_events_normalized.csv",
    index=False
)

print("Normalized baseline saved successfully.")

Normalized baseline saved successfully.


In [ ]:
unified_security_events.to_csv(
    "unified_security_events_clean_v1.csv",
    index=False
)

print("Saved successfully:")
print("unified_security_events_clean_v1.csv")

Saved successfully:
unified_security_events_clean_v1.csv
